# 60 ? Phase 7: Prior Case Retrieval (PCR)

For each query, retrieve **exactly seven distinct candidate case IDs**, ranked by relevance and drawn
only from that split's candidate pool. The final file is an object `{query_id: [ranked candidate IDs]}`
inside root-level `prediction.json` in a ZIP. IDs remain strings so leading zeros survive.

The bundled brief's last defined task is Phase 7 (November 3?7, 2026); Phase 8 is TBA and has no published
interface in this repository. This notebook implements the defined retrieval task.

**Official schema preview verified on 2026-11-07:** JSON objects keyed by case ID; `text` is a list of
sentence strings. Train/dev queries also have `relevant_candidates`; test queries omit that field.
Each split has its own candidate file. No candidate pool is merged across splits.

Primary metric: **Micro-F1@7**. Secondary: micro-precision@7 and micro-recall@7. Compare the top seven
against the **complete** gold set, including queries with more than seven relevant cases.

Brief ?13 requires at least four substantially different experiments, at least three adaptation techniques,
compulsory LoRMA and a smaller-student KD run. The editable table covers full, LoRMA, partial and KD,
using A/B/C/D for breadth. It is a template, not completed experiments or tuned settings.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
%pip -q install numpy pandas tokenizers ijson "huggingface_hub[hf_xet]"
# Install before importing model modules. Restart if pip requests it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, glob, zipfile, tempfile, importlib, shutil, time, json
from pathlib import Path
MYDRIVE = Path('/content/drive/MyDrive')
DRIVE_ROOT = MYDRIVE / 'cs781'
RUNS_DIR = DRIVE_ROOT / 'runs'
OUT_DIR = DRIVE_ROOT / 'pcr_runs'
P7 = Path('/content/p7')
P7.mkdir(exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional explicit folder or archive; leave None to find an updated Phase-7 code copy.
CODE_SOURCE = None
levels = ['', '*/', '*/*/', '*/*/*/']
if CODE_SOURCE is None:
    folders = [Path(h).parents[2] for level in levels
               for h in glob.glob(str(MYDRIVE / f'{level}lm_lib/pcr/batching.py'))]
    archives = [Path(h) for level in levels for h in glob.glob(str(MYDRIVE / f'{level}cs781_code*.zip'))]
    valid_archives = []
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            if 'lm_lib/pcr/batching.py' in z.namelist():
                valid_archives.append(archive)
    candidates = [(p, (p / 'lm_lib/pcr/batching.py').stat().st_mtime) for p in folders]
    candidates += [(p, p.stat().st_mtime) for p in valid_archives]
    if not candidates:
        raise FileNotFoundError('Upload updated lm_lib/, models/, adapters/ folders, or rebuild and upload cs781_code.zip.')
    CODE_SOURCE = max(candidates, key=lambda item: item[1])[0]
CODE_SOURCE = Path(CODE_SOURCE)
if CODE_SOURCE.is_file():
    CODE_DIR = Path(tempfile.mkdtemp(prefix='cs781_phase7_', dir='/content'))
    with zipfile.ZipFile(CODE_SOURCE) as z:
        z.extractall(CODE_DIR)
else:
    CODE_DIR = CODE_SOURCE
assert (CODE_DIR / 'lm_lib/pcr/batching.py').is_file(), 'Selected code copy lacks Phase 7'
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
# Drop project modules cached from a previous phase/archive so imports use this code copy.
for name in list(sys.modules):
    if name in ('lm_lib', 'models', 'adapters') or name.startswith(('lm_lib.', 'models.', 'adapters.')):
        del sys.modules[name]
# Re-run the later import/configuration cells after re-running setup.
print('Code:', CODE_SOURCE, '\nRuns:', OUT_DIR)

In [ ]:
from huggingface_hub import hf_hub_download, login

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    token_file = CODE_DIR / 'hf_token.txt'
    if token_file.is_file():
        return token_file.read_text().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Set HF_TOKEN in Colab Secrets or the environment, or provide hf_token.txt beside the code.') from exc

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
# These paths and the record schema are specified in the bundled brief ?13.3.
cache_dir = DRIVE_ROOT / 'data' / 'phase7'
cache_dir.mkdir(parents=True, exist_ok=True)
for relative in ['Train/train.json', 'Train/train_candidates.json',
                 'Dev/dev.json', 'Dev/dev_candidates.json',
                 'Test/test.json', 'Test/test_candidates.json']:
    name = Path(relative).name
    cached = cache_dir / name
    if not cached.is_file():
        downloaded = hf_hub_download('Exploration-Lab/CS-781-Capstone', f'Phase_7_PCR/{relative}',
                                     repo_type='dataset', token=HF_TOKEN)
        shutil.copyfile(downloaded, cached)
    shutil.copyfile(cached, P7 / name)
    print(name, f'{(P7 / name).stat().st_size / 1e6:.1f} MB')

In [ ]:
PHASE1 = {
    'A': RUNS_DIR / 'A_qwen_dense_full',
    'B': RUNS_DIR / 'B_encdec',
    'C': RUNS_DIR / 'C_gdn_hybrid',
    'D': RUNS_DIR / 'D_mla_moe',
}
for arch, path in PHASE1.items():
    print(arch, path, '| checkpoints:', sorted(p.name for p in path.glob('*.pt')))

EXPERIMENTS = {
    'A_full': dict(arch='A', method='full', method_kwargs={}, run_name='A_full__firstlast2x256'),
    'B_lorma': dict(arch='B', method='lorma', method_kwargs={'r': 8, 'lorma_alpha': 16},
                    run_name='B_lorma__firstlast2x256'),
    'C_partial': dict(arch='C', method='partial', method_kwargs={'k': 4},
                      run_name='C_partial__firstlast2x256'),
    'D_kd': dict(arch='D', method='full', method_kwargs={}, run_name='D_kd_from_A__firstlast2x256',
                 kd_teacher=str(OUT_DIR / 'A_full__firstlast2x256')),
}
# Run A_full before D_kd. Repeat from this cell with a different selection/new run name.
EXPERIMENT = 'C_partial'
selected = EXPERIMENTS[EXPERIMENT].copy()
ARCH = selected.pop('arch')
if ARCH == 'C':
    %pip -q install flash-linear-attention


In [ ]:
import numpy as np
import pandas as pd
from lm_lib.pcr import data as pd_data
rows = []
query_ids = {}
for split in ['train', 'dev', 'test']:
    queries, pool = pd_data.load_split(P7, split, labeled=(split != 'test'))
    query_ids[split] = {q['id'] for q in queries}
    row = dict(split=split, queries=len(queries), candidates=len(pool),
               blank_queries=sum(not q['text'].strip() for q in queries),
               blank_candidates=sum(not d['text'].strip() for d in pool))
    if split != 'test':
        counts = [len(q['relevant_candidates']) for q in queries]
        row.update(mean_relevant=np.mean(counts), queries_without_positives=sum(n == 0 for n in counts),
                   queries_with_more_than_7=sum(n > 7 for n in counts))
    if split in ['dev', 'test']:
        assert len(pool) >= 7, f'{split} needs at least seven candidates'
    print(split, 'query word percentiles [50, 90, 99, max]:',
          np.percentile([len(q['text'].split()) for q in queries], [50, 90, 99, 100]).astype(int))
    print(split, 'candidate word percentiles [50, 90, 99, max]:',
          np.percentile([len(d['text'].split()) for d in pool], [50, 90, 99, 100]).astype(int))
    rows.append(row)
    del queries, pool
assert not (query_ids['train'] & query_ids['dev']), 'Training/dev query IDs overlap'
display(pd.DataFrame(rows))
# Candidate IDs may legitimately recur in different pools; this is not a reason to merge the pools.

The retrieval representation reuses the existing Phase-1 hidden-state extraction: B uses its encoder;
A/C/D use the final normalized features before the tied LM head. Token states are pooled per chunk,
then chunk vectors are pooled and L2-normalized. Query and candidate use the same adapted backbone.
Ranking uses cosine similarity. There is no new trainable projection or classifier architecture.

`first`/`last`/`first_last` select bounded token spans; `spread` samples evenly spaced windows. Selection
uses exact token counts after full tokenization. This is not full-document coverage. Compare legal
information retained by these choices yourself. Empty text retains its ID and gets a BOS representation;
the data overview reports it so you can assess those cases.

Training uses a fixed, seeded set of random negatives from the TRAIN pool. All gold positives are excluded
from negatives, including positives omitted by `max_positives`. Softmax targets are uniform over selected
positives. These are sampled-candidate training losses; evaluation searches the entire official dev pool.
Hard negatives/in-batch negatives are not implemented by this baseline.

Optimization numbers below are inherited starting values. Choose/tune them yourself. Training batches queries and equal-length case chunks, then accumulates query losses. Bulk
inference batches only equal-length chunks, avoiding attention padding errors in the custom backbones.

In [ ]:
from lm_lib.pcr.trainer import PCRConfig, train_pcr, predict_test
cfg = PCRConfig(
    phase1_run=str(PHASE1[ARCH]), phase1_ckpt='best', **selected,
    data_dir=str(P7), out_dir=str(OUT_DIR),
    chunk_len=256, max_chunks=2, strategy='first_last',
    token_pooling='mean', chunk_agg='mean', normalize_text=True,
    negatives=4, max_positives=2, temperature=0.05,
    n_train=0, n_dev=0, embed_batch_size=16, search_batch_size=64,
    train_batch_size=1, train_chunk_batch_size=64, grad_accum=32, grad_checkpoint=True,
    auto_batch=True, batch_mem_frac=0.90,
    effective_batch_queries=32, max_train_batch=32, max_embed_batch=1024,
    epochs=3, lr=5e-5, weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    eval_every=0, kd_alpha=0.5, kd_temperature=2.0, stop_at='', seed=1337,
)
run_dir = OUT_DIR / cfg.run_name
assert not run_dir.exists() or not any(run_dir.iterdir()), 'Use a new run name; optimizer resume is not implemented'
for name in ['best.pt', 'tokenizer.json']:
    assert (PHASE1[ARCH] / name).is_file(), f'Missing Phase-1 file: {PHASE1[ARCH] / name}'
if cfg.kd_teacher:
    assert (Path(cfg.kd_teacher) / 'best.pt').is_file(), 'Train A_full teacher first'
print(cfg)
# The next cell calibrates physical batches; the chosen effective batch remains exactly 32.
# Use effective_batch_queries=4 to match the earlier notebook's effective query batch.
# OOM even at batch 1: reduce chunks/length/negatives, or use an adapter/checkpointing.
# B's chunk length is bounded by its pretrained encoder src_len, not its combined seq_len.
# n_dev > 0 evaluates a saved reproducible query subset against the FULL dev candidate pool.

Batch sizing preserves `effective_batch_queries` through the combination of physical query batches
and gradient accumulation. Each query retains an independent candidate set; this does not add
in-batch negatives. `train_chunk_batch_size` controls equal-length chunks per forward, while
`embed_batch_size` controls candidate encoding. Probes preserve random state, and the training model
is rebuilt from the original Phase-1 weights before optimization. The sizing logic remains enabled
without changing model hyperparameters or using validation scores.


In [ ]:
from lm_lib.pcr.batching import calibrate_batches
cfg = calibrate_batches(cfg)
# cfg.auto_batch is now False, so train_pcr will use these settings without repeating the probes.
batch_report = cfg.batch_calibration
print('Selected train queries/microbatch:', cfg.train_batch_size)
print('Chunk batch cap:', cfg.train_chunk_batch_size)
print('Gradient accumulation:', cfg.grad_accum)
print('Effective queries/step:', cfg.train_batch_size * cfg.grad_accum)
print('Embedding chunks/batch:', cfg.embed_batch_size)
print('Calibration will be saved with the run as batch_calibration.json.')

In [ ]:
# Preflight using the actual selected checkpoint and TRAIN candidate pool.
import gc, torch
import torch.nn.functional as F
from lm_lib.pcr.trainer import build_retriever, ranking_batch_losses, chunks
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tok, report = build_retriever(cfg, DEVICE)
queries, pool = pd_data.load_split(P7, 'train')
pairs, _ = pd_data.sample_pairs(queries[:20], pool, cfg.negatives, cfg.max_positives, cfg.seed)
pair = pairs[0]
qdoc = next(q for q in queries if q['id'] == pair['query_id'])
by_id = {d['id']: d for d in pool}
model.train()
query_tokens = {qdoc['id']: chunks(qdoc, tok, cfg)}
candidate_tokens = {i: chunks(by_id[i], tok, cfg) for i in pair['candidate_ids']}
loss = ranking_batch_losses(model, [pair], query_tokens, candidate_tokens, cfg, DEVICE).mean()
assert torch.isfinite(loss)
loss.backward()
assert any(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0
           for p in model.parameters() if p.requires_grad), 'No useful feature/adapter gradient'
print('Preflight:', DEVICE, report, '| sampled ranking loss:', float(loss.detach()))
print('Sampled candidate IDs:', pair['candidate_ids'], '| positives first:', pair['n_positive'])
del model, tok, queries, pool, pairs, pair, qdoc, by_id, query_tokens, candidate_tokens, loss
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

For retrieval KD, the trained teacher scores the student's sampled TRAIN query/candidate tuples. The
trainer saves raw scores, candidate order and teacher checkpoint hash. The student combines its labeled
ranking loss with KL divergence to the teacher's temperature-scaled score distribution. No dev/test
query relevance or teacher dev/test scores become training targets.

The teacher and student must reference the same official TRAIN data fingerprint, and the student must
have fewer original Phase-1 parameters. Different tokenizers/representations are supported because the
transferred signal is a scalar score per candidate ID. If your D checkpoint is not smaller than A,
select a genuinely smaller Phase-1 student; the pipeline checks real sizes.

Runs save their best checkpoint by dev Micro-F1@7. Use `n_dev=0` for full dev evaluation. The `stop_at`
deadline is checked between accumulation groups; final evaluation/checkpointing takes additional time.
Best weights are inference checkpoints; there is no optimizer resume. Preserve original Phase-1 weights
and tokenizer alongside learned feature/adapter tensors; reload validates their identity.

In [ ]:
results = train_pcr(cfg)
print('Saved:', run_dir)
print('Best dev metrics:', {k: v for k, v in results['best_dev_metrics'].items() if k != 'per_query'})
print('Optimizer steps:', results['optimizer_steps'], '| stopped early:', results['stopped_early'])

In [ ]:
# Inspect held-out rankings and hit counts as evidence for your own analysis.
metrics = json.loads((run_dir / 'dev_metrics.json').read_text())
dev_predictions = json.loads((run_dir / 'dev_predictions.json').read_text())
display(pd.DataFrame(metrics['per_query']).sort_values(['hits', 'id']))
for ident, ranking in list(dev_predictions.items())[:3]:
    print(ident, ranking)
# Gold sets are complete; no padding/truncation of relevant_candidates is applied.

In [ ]:
# Rebuild the selected run and retrieve exclusively from test_candidates.json.
# Exact cache keys prevent reuse after checkpoint, pool text, or representation changes.
zip_path, test_predictions = predict_test(str(run_dir), data_dir=str(P7))
from lm_lib.pcr.metrics import validate_predictions
queries, pool = pd_data.load_split(P7, 'test', labeled=False)
validate_predictions(test_predictions, [q['id'] for q in queries], [d['id'] for d in pool])
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist() == ['prediction.json']
    assert isinstance(json.loads(z.read('prediction.json')), dict)
print('Submission:', zip_path)
print('Preview:', list(test_predictions.items())[:2])
del queries, pool
# Optional download:
# from google.colab import files
# files.download(zip_path)

In [ ]:
rows = []
for path in OUT_DIR.glob('*/results.json'):
    r = json.loads(path.read_text())
    manifest = json.loads((path.parent / 'manifest.json').read_text())
    m = r['best_dev_metrics']
    rows.append(dict(run=r['run'], phase1=Path(r['phase1_run']).name, method=r['method'], kd=bool(r['kd_teacher']),
                     train_batch_size=r.get('train_batch_size', 1),
                     effective_queries=r.get('effective_batch_queries'), embed_batch_size=r.get('embed_batch_size'),
                     micro_f1_at7=m['micro_f1_at7'], micro_precision_at7=m['micro_precision_at7'],
                     micro_recall_at7=m['micro_recall_at7'], n_train=r['n_train'], n_dev=r['n_dev'],
                     base_params=r['base_params'], trainable_params=r['trainable_params'],
                     trainable_pct=r['trainable_pct'],
                     stopped_early=r['stopped_early'],
                     eval_key=(manifest['dev_fingerprint'], tuple(sorted(manifest['eval_query_ids'])))))
comparison = pd.DataFrame(rows)
if not comparison.empty:
    print('Same dev data/query subset:', comparison.eval_key.nunique() == 1)
    display(comparison.drop(columns='eval_key').sort_values('micro_f1_at7', ascending=False))
    comparison.drop(columns='eval_key').to_csv(OUT_DIR / 'comparison.csv', index=False)
    print('Adaptation methods:', sorted(comparison.method.unique()),
          '| LoRMA:', bool((comparison.method == 'lorma').any()), '| KD:', bool(comparison.kd.any()))
print('Check the required experiment count/methods and export a submission for each chosen experiment.')

Retain configurations, manifest/data hashes, selected training pairs, logs, best weights, dev rankings/metrics,
KD targets/provenance, and submission inference records. Test candidate embeddings are cached with exact
pool/checkpoint/representation keys; they may be large. The ZIP itself contains only `prediction.json`.

Write your own conclusions: compare full/partial/LoRMA/KD, an undistilled student, representation coverage,
legal retrieval errors, negatives, parameter counts. The supplied numbers are starting
settings, not tuned recommendations or results. References/assistance must follow brief ?17.5.

Regression tests exercise tiny existing A/B models, actual official schema shapes, leading zeros, complete-gold
@7 metrics, batched feature equivalence, adapters/freezing, KD isolation, checkpoint reload and pool-correct
submission export. 
No leaderboard submission or performance claim is implied by the notebook templates.